# Notebook 04: Inverted Index

## KLEAN Chatbot — Restaurant Information Retrieval & Recommendation

### Overview
An **Inverted Index** is the primary data structure for keyword-based Information Retrieval. It maps every unique term in the vocabulary to the list of document IDs (called a **Postings List**) where that term occurs, along with its within-document Term Frequency (TF).

**Core Objectives:**
1. Build the inverted index from the preprocessed 698-restaurant corpus using `InvertedIndex` from `src.indexing.inverted_index`.
2. Inspect index metrics: vocabulary size, postings list length distribution, and average document length.
3. Perform term lookups for both English terms (`"pizza"`, `"soup"`, `"bkk1"`) and Khmer terms (`"មី"`, `"បាយ"`).
4. Test document lookup, addition, update, and removal.
5. Save and reload the serialized index.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.data_loader import DataLoader
from src.utils.document_converter import DocumentConverter
from src.preprocessing.text_preprocessing import TextPreprocessor
from src.indexing.inverted_index import InvertedIndex

print(f"Project root: {PROJECT_ROOT}")

Project root: D:\Year4\Web_Retrive\Project


## 1. Load Data, Convert Documents, and Preprocess

In [2]:
# Load records and convert to documents
loader = DataLoader(PROJECT_ROOT / "dataset" / "restaurants.json")
records = loader.load()

converter = DocumentConverter()
documents = converter.convert_all(records)
corpus = converter.get_corpus()

# Preprocess
preprocessor = TextPreprocessor()
tokenized_corpus = preprocessor.preprocess_documents(corpus)

print(f"Corpus size: {len(tokenized_corpus)} tokenized documents")

| 2026-11-24 00:17:51,806 | INFO | khmer-nltk | Loaded model from c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\khmernltk\word_tokenize\sklearn_crf_ner_10000.sav |


Corpus size: 698 tokenized documents


## 2. Build the Inverted Index

In [3]:
inv_index = InvertedIndex()
inv_index.build(tokenized_corpus)

print("=== Inverted Index Statistics ===")
print(f"Total Indexed Documents : {inv_index.num_documents:,}")
print(f"Total Vocabulary Size   : {inv_index.vocabulary_size:,} unique terms")
print(f"Average Document Length : {inv_index.avg_doc_length:.2f} tokens")

=== Inverted Index Statistics ===
Total Indexed Documents : 698
Total Vocabulary Size   : 2,399 unique terms
Average Document Length : 20.39 tokens


## 3. Query Postings Lists (Term Lookup)

Lookup postings lists and term frequencies across English and Khmer terms.

In [4]:
test_terms = ["soup", "noodles", "pizza", "coffee", "bkk1", "មី", "បាយ", "ហាង"]

for term in test_terms:
    postings = inv_index.get_postings(term)
    df_count = inv_index.get_document_frequency(term)
    sample_docs = postings[:5]
    print(f"Term: '{term:10s}' | DF: {df_count:3d} docs | Sample Postings: {sample_docs}")

Term: 'soup      ' | DF: 138 docs | Sample Postings: [22, 33, 34, 37, 44]
Term: 'noodles   ' | DF:  36 docs | Sample Postings: [79, 99, 104, 134, 151]
Term: 'pizza     ' | DF:   1 docs | Sample Postings: [259]
Term: 'coffee    ' | DF:  31 docs | Sample Postings: [6, 23, 28, 30, 76]
Term: 'bkk1      ' | DF:   1 docs | Sample Postings: [152]
Term: 'មី        ' | DF:   9 docs | Sample Postings: [510, 556, 557, 558, 560]
Term: 'បាយ       ' | DF:  33 docs | Sample Postings: [12, 25, 108, 420, 442]
Term: 'ហាង       ' | DF:  16 docs | Sample Postings: [141, 643, 644, 645, 646]


## 4. Inspect Document Details from Postings

Verify that the restaurants retrieved from the postings list genuinely match the query concept.

In [5]:
# Query for Khmer noodle term 'មី'
noodle_doc_ids = inv_index.get_postings("មី")
print(f"Total restaurants matching 'មី': {len(noodle_doc_ids)}")

for doc_id in noodle_doc_ids[:3]:
    rec = converter.get_record(doc_id)
    tf = inv_index.get_term_frequency("មី", doc_id)
    print(f" - [Doc {doc_id}] {rec['name']} | Category: {rec['category']} | TF: {tf}")

Total restaurants matching 'មី': 9
 - [Doc 510] ផ្ទះមីប៉ក់ - Mee Pok House | Category: Soup Restaurant | TF: 1
 - [Doc 556] មីគោកពិសេសអូរឬស្សី | Category: Breakfast And Brunch Restaurant | TF: 1
 - [Doc 557] មីឆាលើអាកាស | Category: Noodles Restaurant | TF: 1


## 5. Document Lookup & Forward Index Inspection

Inspect all terms and their term frequencies contained within a given document.

In [6]:
doc_id = 1
terms_in_doc = inv_index.get_document_terms(doc_id)
print(f"Terms in Document {doc_id} (total unique {len(terms_in_doc)}):")
print(terms_in_doc)

Terms in Document 1 (total unique 15):
{'name': 1, '13eo': 2, 'breakfast': 3, 'category': 2, 'brunch': 1, 'restaurant': 2, 'sub': 1, 'vietnamese': 1, 'tea': 1, 'room': 1, 'address': 1, 'ភ្នំពេញ': 1, 'postal': 1, 'code': 1, '121201': 1}


## 6. Dynamic Index Updates (Add, Update, Remove)

Demonstrate updating the index without rebuilding it from scratch.

In [7]:
# 1. Add new restaurant
new_doc_id = 9999
new_tokens = ["artisan", "gelato", "dessert", "bkk1"]
inv_index.add_document(new_doc_id, new_tokens)
print(f"Added Doc 9999. Postings for 'gelato': {inv_index.get_postings('gelato')}")

# 2. Update restaurant
updated_tokens = ["artisan", "gelato", "sorbet", "bkk1"]
inv_index.update_document(new_doc_id, updated_tokens)
print(f"Updated Doc 9999. Postings for 'sorbet': {inv_index.get_postings('sorbet')}")
print(f"Postings for 'dessert' (should now be empty): {inv_index.get_postings('dessert')}")

# 3. Remove restaurant
inv_index.remove_document(new_doc_id)
print(f"Removed Doc 9999. Postings for 'gelato' after removal: {inv_index.get_postings('gelato')}")

Added Doc 9999. Postings for 'gelato': [9999]
Updated Doc 9999. Postings for 'sorbet': [9999]
Postings for 'dessert' (should now be empty): []
Removed Doc 9999. Postings for 'gelato' after removal: []


## 7. Index Serialization & Persistence

Save the inverted index to disk and test reloading it.

In [8]:
index_path = PROJECT_ROOT / "outputs" / "inverted_index.json"
inv_index.save(index_path)
print(f"Saved Inverted Index to: {index_path}")

# Reload
loaded_index = InvertedIndex.load(index_path)
print(f"Loaded Index Terms: {loaded_index.vocabulary_size:,} | Docs: {loaded_index.num_documents:,}")
assert loaded_index.get_postings("soup") == inv_index.get_postings("soup")
print("Serialization roundtrip verified!")

Saved Inverted Index to: D:\Year4\Web_Retrive\Project\outputs\inverted_index.json
Loaded Index Terms: 2,399 | Docs: 698
Serialization roundtrip verified!


## 8. Summary & Next Steps
- The **Inverted Index** is built and verified with instant term lookups across 698 Phnom Penh restaurants.
- Both English terms (`"soup"`, `"pizza"`, `"bkk1"`) and Khmer terms (`"មី"`, `"បាយ"`) are mapped to their postings lists and term frequencies.
- Dynamic index updates (add, update, delete) and JSON persistence work seamlessly.
- **Next up (Phase 3)**: Boolean Retrieval (AND, OR, NOT), TF-IDF, and BM25 ranking algorithms directly powered by this inverted index!